# 【講師用】Ch.1 データ読み込み・集計（pandas）完全解説

> 🔒 **このノートブックは講師専用です。受講者には配布しないでください。**

---

## 📢 講師向け冒頭ガイド

### このチャプターで伝えること
- データ分析は「コードを書くこと」ではなく「データを理解すること」が本質
- pandas は「大きな Excel」だと思えばよい。Excelでできることは全部できる
- 欠損値の処理は「正解がない」問題。判断の軸を持つことが重要

### よくある受講者の躓きポイント
| 躓き | 対処法 |
|------|--------|
| `groupby` の書き方が覚えられない | 「AIに書いてもらえばいい。結果の解釈ができれば十分」と伝える |
| `fillna` と `dropna` どちらを使うか | 「欠損率が5%以下なら削除も選択肢。実務では上司やチームと相談して決める」と伝える |
| `isnull().sum()` の意味が分からない | `isnull()` が True/False の表を返し、`sum()` が True=1 を合計するとイメージで説明 |

### 時間配分目安（座学20分）
- 1.1 データ読み込みと基本確認：8分
- 1.2 欠損値の確認と処理：7分
- 1.3 フィルタリングと集計：5分

---
## 1.1 データの読み込みと基本確認

### 📢 講師メモ
ここでは「データを手に入れたら最初にやること」を体で覚えてもらいます。
`head()` → `info()` → `describe()` の3点セットは、今後どんなデータを触っても最初に必ず実行するルーティンとして身につけてもらいましょう。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine

# scikit-learn の内蔵データセットを読み込む
wine = load_wine()

# wine は辞書に似た「Bunch」オブジェクト
# wine.data        : 数値データ（numpy array）
# wine.feature_names: 列名のリスト
# wine.target      : ラベル（0/1/2）

# pandas の DataFrame に変換する
df = pd.DataFrame(wine.data, columns=wine.feature_names)

# ターゲット（品種番号）と品種名を追加する
df['target'] = wine.target
df['品種'] = df['target'].map({0: 'Barolo', 1: 'Grignolino', 2: 'Barbera'})

print('データ件数:', len(df))
print('列数:', len(df.columns))

### 📢 `map()` の説明
```python
df['target'].map({0: 'Barolo', 1: 'Grignolino', 2: 'Barbera'})
```
> `map()` は「辞書を使って値を別の値に変換する」メソッドです。
> 数字の 0 → 'Barolo' に置き換えるイメージです。
> 実務では「コード → 名称」変換によく使います（例：都道府県コード → 都道府県名）。

In [ ]:
# 【ステップ1】先頭5行を確認する
# → 列名・値の形式・データの「顔」を掴む
df.head()

### 📢 `head()` で確認すること
- 列名が英語か日本語か
- 数値列か文字列列か（後で `info()` で確認）
- 値が 0〜1 の範囲か、大きな数値か（スケーリングの必要性を判断）

In [ ]:
# 【ステップ2】データ型と件数を確認する
# → 型が正しいか・欠損値がないかを確認する
df.info()

### 📢 `info()` で確認すること
- `Non-Null Count` が `178` 以外の列 → 欠損あり
- `Dtype` が `object` の列 → 文字列（機械学習前に数値化が必要）
- `float64` / `int64` は数値列 → そのまま使える

> 💡 `info()` は「データの骨格チェック」です。欠損値は `isnull().sum()` で再確認します。

In [ ]:
# 【ステップ3】基本統計量を確認する
# → 平均・最大・最小・標準偏差から「おかしな値」を見つける
df.describe()

### 📢 `describe()` で確認すること
- `min` と `max` の差が非常に大きい列 → スケーリングが必要（Ch.4で登場）
- `min` が負の値 → 欠損を 0 で埋めると問題が起きることがある
- 今回の `proline`：min=278 / max=1680 と `alcohol`：min=11.0 / max=14.8 では桁が全く違う

> 「このスケールの差を放置すると機械学習ではどうなるか？」という問いを Ch.4 に繋げてください。

---
## 1.2 欠損値の確認と処理

### 📢 講師メモ
「欠損値はどう処理するのが正解ですか？」という質問が必ず出ます。
**「正解はない」** と明確に伝えましょう。判断の軸は以下の2つです：
1. 欠損率（5%以下なら削除も現実的）
2. データの性質（ランダムな欠損か、システム的な欠損か）

In [ ]:
# 演習用：欠損値を人工的に追加する
# （実際のワインデータには欠損値がないため、練習のために意図的に作る）
df_with_na = df.copy()  # 元のデータを壊さないようにコピー
np.random.seed(42)      # 再現性のための乱数シード固定

for col in ['alcohol', 'color_intensity', 'proline']:
    # 各列の10件をランダムに NaN に置き換える
    idx = np.random.choice(df_with_na.index, size=10, replace=False)
    df_with_na.loc[idx, col] = np.nan

print('各列の欠損値件数：')
print(df_with_na.isnull().sum())

### 📢 `isnull().sum()` の仕組み
```python
df.isnull()      # True/False の表を返す（欠損 = True）
.sum()           # True = 1 として合計する = 欠損件数
```
> 「`True` が `1` として扱われる」というPythonの性質を活用しています。
> `.mean()` にすると欠損率（0.0〜1.0）が出ます。

In [ ]:
# 欠損率（%）を計算する
# → 欠損件数 ÷ 全件数 × 100
missing_rate = df_with_na.isnull().sum() / len(df_with_na) * 100
print('欠損率（%）：')
print(missing_rate[missing_rate > 0].round(1))

### 📢 欠損率 5.6% はどう判断するか
- 一般的には **5〜10%以下** であれば「削除」「平均補完」どちらも現実的
- 20%以上になると、どう補完するかをより慎重に考える必要がある
- 「欠損がなぜ発生したか」の原因を探ることが本来は重要（センサー故障？入力漏れ？）

In [ ]:
# ─── 方法1：平均値で補完（fillna）─────────────────────────────
# 最もよく使われる補完方法。全体の平均で埋める。

df_filled = df_with_na.copy()
for col in ['alcohol', 'color_intensity', 'proline']:
    mean_val = df_filled[col].mean()       # 欠損を除いた平均を計算
    df_filled[col] = df_filled[col].fillna(mean_val)   # 欠損を平均値で埋める
    print(f'{col}: 平均値 {mean_val:.2f} で補完')

print('\n補完後の欠損値件数：', df_filled.isnull().sum().sum())

### 📢 `fillna(平均値)` の問題点
- **全体の平均** で埋めると、品種ごとの特性が薄まる可能性がある
- 例：Barolo のアルコール度数の欠損を「全品種の平均」で補完すると、
  Barolo 本来の特性（高アルコール）が損なわれる
- → 演習問4では **品種ごとの平均** で補完する方法を学ぶ

In [ ]:
# ─── 方法2：欠損値のある行を削除（dropna）───────────────────────
# シンプルだが、データ量が減るデメリットがある

df_dropped = df_with_na.dropna()
print('削除前:', len(df_with_na), '件')
print('削除後:', len(df_dropped), '件')
print(f'削除件数: {len(df_with_na) - len(df_dropped)} 件')

### 📢 `dropna` を使う際の注意点
- 1列でも欠損があれば **その行全体** を削除する（デフォルト）
- 特定の列だけを対象にしたい場合：`df.dropna(subset=['alcohol'])`
- 削除後にインデックスがズレるので、必要なら `reset_index(drop=True)` を忘れずに

---
## 1.3 フィルタリングと集計

### 📢 講師メモ
「Excelのフィルター機能と同じです」と伝えると理解が早まります。
`groupby` は「ピボットテーブル」のイメージで説明しましょう。

In [ ]:
# 品種ごとの件数を確認
# value_counts()：ユニークな値ごとの出現回数を集計する
print('品種ごとの件数：')
print(df['品種'].value_counts())

In [ ]:
# ─── フィルタリング ───────────────────────────────────────────────
# 「アルコール度数が 13.0 以上」のデータだけ抽出する
#
# 仕組み：
#   df['alcohol'] >= 13.0  → True/False の Series を返す
#   df[True/Falseの配列]   → True の行だけを取り出す

high_alcohol = df[df['alcohol'] >= 13.0]
print(f'アルコール度数13以上: {len(high_alcohol)} 件')
high_alcohol.head()

### 📢 複数条件フィルタの書き方
```python
# AND 条件：& を使い、各条件を () で囲む
df[(df['alcohol'] >= 13.0) & (df['color_intensity'] >= 5.0)]

# OR 条件：| を使う
df[(df['alcohol'] >= 13.0) | (df['color_intensity'] >= 5.0)]

# ⚠️ and / or は使えない！必ず & / | を使うこと
```
> 括弧 `()` を忘れると演算子の優先順位でエラーになります。これは受講者がよくやるミスです。

In [ ]:
# ─── groupby：グループごとに集計 ─────────────────────────────────
# 品種ごとに「アルコール度数・色の濃さ・プロリン」の平均を集計する
#
# Excelのピボットテーブルと同じ操作！

group_mean = df.groupby('品種')[['alcohol', 'color_intensity', 'proline']].mean().round(2)
print('品種ごとの平均値：')
group_mean

### 📢 `groupby` の結果から読み取れること
- **Barolo**：アルコール度数が最も高く、色が最も濃い。プロリンも圧倒的に多い
- **Barbera**：色が比較的濃いが、アルコール・プロリンは中程度
- **Grignolino**：アルコール・色・プロリンすべてが低め

> 💡 この「品種ごとの違い」が、機械学習で分類できる理由です。Ch.4 で繋がります。

In [ ]:
# ─── sort_values：並べ替え ────────────────────────────────────────
# 色の濃さが高い順に並べる（ascending=False で降順）

df_sorted = df.sort_values('color_intensity', ascending=False)
df_sorted[['品種', 'alcohol', 'color_intensity']].head(10)

---
## 🔑 演習の解答・解説

### 📢 演習進行のポイント
- 受講者に最低10分は自力でやらせてから解説に移る
- 「AIにコードを生成させた場合」「自分で書いた場合」どちらでも、結果の解釈が言えるかを確認する
- 問4（発展）は時間があれば。`groupby + transform` の概念を理解するだけでOK

### 問1：品種ごとのデータ件数を確認する

**解説のポイント：** `value_counts()` はカテゴリ変数の分布を最速で確認するメソッド。データが偏っていないか確認するために使います。

In [ ]:
# ✅ 問1：解答
print('品種ごとのデータ件数：')
print(df['品種'].value_counts())

print('\n割合（%）：')
print((df['品種'].value_counts(normalize=True) * 100).round(1))

**📢 解説：**
- Barolo: 59件、Grignolino: 71件、Barbera: 48件
- 最大（Grignolino）と最小（Barbera）で1.5倍弱の差はあるが、極端な不均衡ではない
- 「不均衡データ」とは一方が90%以上を占めるような場合を指す（Ch.5 の乳がんデータで体験）

### 問2：プロリン含有量が最も多い品種を特定する

**解説のポイント：** `groupby → mean → sort_values` のチェーンは実務でも毎日使うパターン。

In [ ]:
# ✅ 問2：解答
proline_by_type = df.groupby('品種')['proline'].mean().sort_values(ascending=False)
print('品種ごとのプロリン平均：')
print(proline_by_type.round(1))

print(f'\n最もプロリンが多い品種: {proline_by_type.idxmax()}')

**📢 解説：**
- Barolo が圧倒的に多い（約1100）
- Grignolino と Barbera は 500〜600 程度で似ている
- プロリンは Ch.4 の特徴量重要度でも上位に来る → EDA での発見がモデルで検証される流れを予告

### 問3：アルコール度数が高く、色が濃いワインを絞り込む

**解説のポイント：** AND 条件のフィルタ。括弧忘れが最多エラーなので注意を促す。

In [ ]:
# ✅ 問3：解答
filtered = df[(df['alcohol'] >= 13.5) & (df['color_intensity'] >= 5.0)]

print(f'条件を満たすワイン: {len(filtered)} 件')
print('\n品種の内訳：')
print(filtered['品種'].value_counts())

# 品種の割合も確認
print('\n割合（%）：')
print((filtered['品種'].value_counts(normalize=True) * 100).round(1))

**📢 解説：**
- 条件を満たすほとんどが **Barolo** であることがわかる
- つまり「アルコール度数が高くて色が濃い → Barolo の可能性が高い」という特性がある
- これが Ch.4 の機械学習で「分類」できる根拠になっている → 「EDA で見えたことをモデルが学ぶ」と繋げる

### 問4（発展）：欠損値補完を品種ごとの平均に変更する

**解説のポイント：** `groupby + transform` は少し高度。「なぜ全体平均より品種別平均が良いか」の概念を理解できれば十分。

In [ ]:
# ✅ 問4：解答
# groupby + transform の仕組み：
# - groupby('品種') で品種別にグループを作る
# - transform('mean') で「各行の品種グループの平均」を返す
# → 返ってくるのは元の DataFrame と同じ行数 = fillna に直接渡せる

df_filled_group = df_with_na.copy()

# 品種ごとの平均でアルコール度数の欠損を補完
group_means = df_filled_group.groupby('品種')['alcohol'].transform('mean')
df_filled_group['alcohol'] = df_filled_group['alcohol'].fillna(group_means)

print('補完後の確認（補完前に欠損があった行）：')
# 欠損があった行を確認
was_missing = df_with_na['alcohol'].isnull()
print(df_filled_group.loc[was_missing, ['品種', 'alcohol']].head(10))

**📢 解説：**
- 品種が `Barolo` の欠損行には Barolo の平均値が入り、`Grignolino` の欠損行には Grignolino の平均値が入る
- 全体平均との差：全体平均は約 13.0 だが、Barolo の平均は約 13.7 → より実態に即した補完ができる

**`transform` vs `agg` の違い（補足）：**
```python
# agg：グループごとに1行にまとめる（元のインデックスとは別）
df.groupby('品種')['alcohol'].agg('mean')   # 3行だけ返ってくる

# transform：元と同じ行数で返す（fillna に直接使える）
df.groupby('品種')['alcohol'].transform('mean')  # 178行返ってくる
```

---
## ✅ チャプターのまとめ（講師用コメント付き）

| 操作 | コード | 覚えるポイント |
|------|--------|---------------|
| データ読み込み | `pd.DataFrame(data, columns=columns)` | 列名リストを忘れずに |
| 基本確認 | `head()`, `info()`, `describe()` | この3つは必ずセットで |
| 欠損値確認 | `isnull().sum()` | まず確認してから方針を決める |
| 欠損値補完 | `fillna(value)` | 平均・中央値・最頻値など選択肢は多い |
| フィルタリング | `df[条件式]` | AND は `&`、括弧 `()` を忘れない |
| グループ集計 | `groupby('列名').mean()` | Excel のピボットテーブルと同じ |
| ソート | `sort_values('列名', ascending=False)` | `ascending=False` で降順 |

### 📢 Ch.2 への橋渡し
> 「数値で集計した内容を、次のチャプターではグラフで視覚化します。
> Barolo が他の品種と違うことは数字でわかりましたが、グラフにすると一目でわかるようになります。」